# Verification notebook for Sign uncertainty principle

This notebook is a self-contained mathematical and computational companion to the Sign uncertainty principle section in the paper.

The task template, including the prompt given to the agents, is available [here](https://github.com/dualverse-ai/station/tree/main/example/research_alpha_evolve/uncertainty).

The outline is as follows:

- [1. Problem Definitions](#/notebooks/sign_uncertainty/verification.ipynb/section-1)
- [2. S1. A new upper bound of $0.3089$.](#/notebooks/sign_uncertainty/verification.ipynb/section-2) ([Formal proof](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/sign_uncertainty/formal_proofs/s1))
- [3. S2. The double-root Laguerre family is exhausted near $0.3153$.](#/notebooks/sign_uncertainty/verification.ipynb/section-3) ([Formal proof](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/sign_uncertainty/formal_proofs/s2))
- [4. Additional findings](#/notebooks/sign_uncertainty/verification.ipynb/section-4)

The required packages are:

- `mpmath`
- `SymPy`
- `cypari2`

<a id="section-1"></a>

## 1. Problem Definitions

The one-dimensional sign-uncertainty problem asks how soon a function and its Fourier transform can both become permanently nonnegative when both are negative at the origin.

We use

$$
\widehat f(\xi)=\int_{\mathbb R}f(x)e^{-2\pi i x\xi}\,dx,
\qquad
A(f)=\inf\{r>0:f(x)\ge0\text{ for }|x|\ge r\}.
$$

The one-dimensional sign-uncertainty constant is

$$
C_{\rm SU}=\inf_f A(f)A(\widehat f),
$$

where $f$ is nonzero, real and even, both $f$ and $\widehat f$ are integrable and eventually nonnegative, and $f(0),\widehat f(0)<0$.

**Lemma 1.1 (Laguerre--Gaussian Fourier eigenfunctions).** For $n\ge0$,

$$
\phi_n(x)=L_n^{(-1/2)}(2\pi x^2)e^{-\pi x^2}
\quad\text{satisfies}\quad
\widehat\phi_n=(-1)^n\phi_n.
$$

Consequently, if $P$ is a linear combination of $L_0^{(-1/2)},L_2^{(-1/2)},\ldots$, then $P(2\pi x^2)e^{-\pi x^2}$ is self-Fourier.

**Proof.** The Hermite--Laguerre identity expresses $\phi_n$ as a nonzero scalar multiple of the even Hermite function of order $2n$. Under the stated Fourier normalization, the Hermite function of order $m$ has eigenvalue $(-i)^m$. Taking $m=2n$ gives $(-i)^{2n}=(-1)^n$. $\square$

**Lemma 1.2 (from a polynomial tail to sign uncertainty).** Suppose

$$
P(t)=\sum_j a_jL_{2j}^{(-1/2)}(t),\qquad P(0)=0,\qquad P'(0)=1,
$$

and $P(t)<-\varepsilon$ for all $t\ge T$, where $\varepsilon>0$. Then

$$
f_\varepsilon(x)=\bigl(-P(2\pi x^2)-\varepsilon\bigr)e^{-\pi x^2}
$$

is self-Fourier, satisfies $f_\varepsilon(0)=\widehat f_\varepsilon(0)=-\varepsilon$, and is nonnegative for $|x|\ge\sqrt{T/(2\pi)}$. Hence $C_{\rm SU}\le T/(2\pi)$.

**Proof.** Adding the constant $\varepsilon$ changes only the $L_0^{(-1/2)}$ coefficient, so Lemma 1.1 gives self-Fourier invariance. The origin and tail statements follow directly from the displayed formula. $\square$

For a polynomial with tail zeros, direct Gaussian subtraction can reopen negative dips. The following closure argument handles the prescribed-double-root constructions.

**Lemma 1.3 (strict-origin closure).** Let $h=\widehat h$ be a nonzero polynomial--Gaussian with $h(0)=0$ and $h(x)\ge0$ for $|x|\ge R$. Then $C_{\rm SU}\le R^2$.

**Proof.** Choose $a>1$ so that no two nonzero tail zeros of $h$ have squared ratio $a^2$; only finitely many values are excluded. The function

$$
h_a(x)=h(ax)+a^{-1}h(x/a)
$$

is self-Fourier by the Fourier scaling rule. For $|x|\ge aR$, both summands are nonnegative and, by the choice of $a$, never vanish simultaneously. Moreover $h_a(x)e^{\pi x^2}\to+\infty$ as $|x|\to\infty$. Thus $h_a-\delta e^{-\pi x^2}$ is eventually nonnegative and negative at the origin for some $\delta>0$. Its sign radius is at most $aR$. Letting $a\downarrow1$ proves the claim. $\square$

The shared setup below is used by all verification cells. It authenticates and loads the exact data bundle, initializes exact real-root counting, and defines the assertion helper used below.

In [1]:
from __future__ import annotations

import hashlib
import gzip
import json
import math
import sys
from functools import lru_cache
from pathlib import Path

import mpmath as mp
import sympy as sp
from cypari2 import Pari

try:
    sys.set_int_max_str_digits(0)
except AttributeError:
    pass

DATA_PATH = Path("uncertainty_data.json.gz")
EXPECTED_DATA_SHA256 = "04440bb9d0239a68ed38b7d780462380c2b698fdf54f0917c628ff7df35b9826"
raw_data = DATA_PATH.read_bytes()
actual_data_sha256 = hashlib.sha256(raw_data).hexdigest()
assert actual_data_sha256 == EXPECTED_DATA_SHA256
data = json.loads(gzip.decompress(raw_data))
PARI = Pari()
PARI.allocatemem(512 * 1024 * 1024)

def passed(name, value=True):
    assert bool(value), name

passed("data_file_sha256")
print({"file": DATA_PATH.name, "bytes": len(raw_data), "sha256": actual_data_sha256})


PARI stack size set to 536870912 bytes, maximum size set to 536870912
{'file': 'uncertainty_data.json.gz', 'bytes': 14595505, 'sha256': '04440bb9d0239a68ed38b7d780462380c2b698fdf54f0917c628ff7df35b9826'}


<a id="section-2"></a>

## 2. S1. A new upper bound of $0.3089$.

**Theorem 2.1.**

$$
C_{\rm SU}\le0.3089.
$$

Combining $C_{\rm SU}\le0.3089$ with the known lower bound gives

$$
0.2025\le C_{\rm SU}\le0.3089.
$$

**Verification.** The upper bound follows from the rational degree-$226$
polynomial stored in the data file, using exact basis reconstruction, exact normalization, a
Bernstein-basis proof on a bounded interval, and an exact shifted-power proof on the remaining ray.

For a polynomial $q(s)=\sum_{j=0}^n\beta_j\binom nj s^j(1-s)^{n-j}$ on $0\le s\le1$, convexity gives $q(s)\le\max_j\beta_j$. Thus nonpositive Bernstein coefficients prove nonpositivity on the whole interval.

The next cell reconstructs the degree-$226$ polynomial in the even Laguerre basis and checks its exact normalization, leading sign, and coefficient hash.

In [2]:
ALPHA = sp.Rational(-1, 2)

def _trim(values):
    values = list(values)
    while len(values) > 1 and values[-1] == 0:
        values.pop()
    return tuple(values)

def _add(left, right):
    out = [sp.Rational(0)] * max(len(left), len(right))
    for i, value in enumerate(left):
        out[i] += value
    for i, value in enumerate(right):
        out[i] += value
    return _trim(out)

def _scale(values, factor):
    return _trim(sp.Rational(factor) * value for value in values)

@lru_cache(maxsize=None)
def laguerre_power_coefficients(degree):
    # Ascending coefficients of L_degree^(-1/2), computed by recurrence.
    if degree == 0:
        return (sp.Integer(1),)
    if degree == 1:
        return (ALPHA + 1, sp.Integer(-1))
    previous = (sp.Integer(1),)
    current = (ALPHA + 1, sp.Integer(-1))
    for n in range(1, degree):
        term = _scale(current, sp.Rational(2*n + 1) + ALPHA)
        term = _add(term, _scale((sp.Integer(0),) + current, -1))
        term = _add(term, _scale(previous, -(sp.Rational(n) + ALPHA)))
        previous, current = current, _scale(term, sp.Rational(1, n + 1))
    return current

def compose_laguerre(degrees, coefficients):
    out = [sp.Rational(0)] * (max(degrees) + 1)
    for degree, coefficient in zip(degrees, coefficients):
        for exponent, value in enumerate(laguerre_power_coefficients(int(degree))):
            out[exponent] += coefficient * value
    return out

continuum = data["continuum"]
degrees = list(map(int, continuum["basis_degrees"]))
laguerre_coeffs = [sp.Rational(x) for x in continuum["laguerre_coefficients"]]
power = [sp.Rational(x) for x in continuum["power_coefficients_ascending"]]
recomposed = compose_laguerre(degrees, laguerre_coeffs)
passed("continuum_exact_basis_recomposition", recomposed == power)
passed("continuum_degree_226", len(power) - 1 == 226 and degrees == list(range(0, 228, 2)))
passed("continuum_P0_zero", power[0] == 0)
passed("continuum_Pprime0_one", power[1] == 1)
passed("continuum_negative_leading_coefficient", power[-1] < 0)

coefficient_block = json.dumps(
    {
        "laguerre_coefficients_rational": [str(x) for x in laguerre_coeffs],
        "power_coefficients_rational_ascending": [str(x) for x in power],
    }, sort_keys=True, separators=(",", ":")
)
coefficient_sha = hashlib.sha256(coefficient_block.encode()).hexdigest()
passed("continuum_coefficient_hash", coefficient_sha == continuum["source_coefficient_sha256"])

print({
    "degree": len(power)-1,
    "even Laguerre modes": len(degrees),
    "P(0)": str(power[0]),
    "P'(0)": str(power[1]),
    "leading sign": "negative",
    "coefficient SHA-256": coefficient_sha[:16] + "…",
})


{'degree': 226, 'even Laguerre modes': 114, 'P(0)': '0', "P'(0)": '1', 'leading sign': 'negative', 'coefficient SHA-256': '8df170dcfc89ea3f…'}


The next cell proves the polynomial tail is negative on $[1.9408,1000]$ by interval Bernstein subdivision and on $[1000,\infty)$ by exact shifted coefficients.

In [3]:
mp.iv.dps = 120

def iv_rational(value):
    value = sp.Rational(value)
    return mp.iv.mpf(int(value.p)) / mp.iv.mpf(int(value.q))

def iv_upper(value):
    return mp.mpf(value.b)

def bernstein_coefficients(power_coeffs, left, right):
    # Outward-rounded Bernstein coefficients on [left,right].
    n = len(power_coeffs) - 1
    a = iv_rational(left)
    h = iv_rational(right - left)
    a_powers = [iv_rational(1)]
    h_powers = [iv_rational(1)]
    for _ in range(n):
        a_powers.append(a_powers[-1] * a)
        h_powers.append(h_powers[-1] * h)
    q_power = []
    for ell in range(n + 1):
        total = iv_rational(0)
        for m in range(ell, n + 1):
            total += (iv_rational(power_coeffs[m]) * math.comb(m, ell)
                      * a_powers[m-ell])
        q_power.append(total * h_powers[ell])
    beta = []
    for j in range(n + 1):
        total = iv_rational(0)
        for ell in range(j + 1):
            total += q_power[ell] * iv_rational(
                sp.Rational(math.comb(j, ell), math.comb(n, ell))
            )
        beta.append(total)
    return beta

def split_bernstein(beta):
    half = iv_rational(sp.Rational(1, 2))
    work = list(beta)
    left, right = [work[0]], [work[-1]]
    for _ in range(1, len(beta)):
        work = [(work[i] + work[i+1]) * half for i in range(len(work)-1)]
        left.append(work[0])
        right.append(work[-1])
    return left, list(reversed(right))

def certify_interval(power_coeffs, left, right, max_depth=27):
    initial = bernstein_coefficients(power_coeffs, left, right)
    stack = [(left, right, initial, 0)]
    accepted = []
    while stack:
        a, b, beta, depth = stack.pop()
        upper = max(iv_upper(value) for value in beta)
        if upper <= 0:
            accepted.append((a, b, depth, upper))
            continue
        assert depth < max_depth, (a, b, upper)
        beta_left, beta_right = split_bernstein(beta)
        middle = (a + b) / 2
        stack.append((middle, b, beta_right, depth + 1))
        stack.append((a, middle, beta_left, depth + 1))
    return accepted

T_RATIONAL = sp.Rational(1213, 625)  # 1.9408
X_MAX = sp.Integer(1000)
EPSILON = sp.Rational(1, 10**6)
bounded_certificate = certify_interval(power, T_RATIONAL, X_MAX)
worst_upper = max(row[3] for row in bounded_certificate)
passed("continuum_Bernstein_tail", worst_upper + mp.mpf("1e-6") < 0)

# Exact coefficients of P(1000+u).
shifted = [
    sum(power[m] * sp.binomial(m, r) * X_MAX**(m-r)
        for m in range(r, len(power)))
    for r in range(len(power))
]
passed("continuum_exact_ray_closure", all(value < 0 for value in shifted))
passed("continuum_epsilon_preserves_ray", shifted[0] + EPSILON < 0)

print({
    "interval": "[1.9408, 1000]",
    "Bernstein subintervals": len(bounded_certificate),
    "largest coefficient upper bound": mp.nstr(worst_upper, 10),
    "shifted coefficients at 1000": len(shifted),
    "positive shifted coefficients": sum(1 if bool(value > 0) else 0 for value in shifted),
    "epsilon": float(EPSILON),
})


{'interval': '[1.9408, 1000]', 'Bernstein subintervals': 27, 'largest coefficient upper bound': '-1.606941512e-5', 'shifted coefficients at 1000': 227, 'positive shifted coefficients': 0, 'epsilon': 1e-06}


The next cell gives rational bounds for $\pi$ and checks that the certified tail threshold implies the stated decimal bound.

In [4]:
def arctan_inverse_bounds(q, terms):
    # Exact alternating-series bounds for arctan(1/q).
    partial = sum(
        sp.Rational((-1)**j, (2*j + 1) * q**(2*j + 1))
        for j in range(terms)
    )
    next_term = sp.Rational((-1)**terms, (2*terms + 1) * q**(2*terms + 1))
    return min(partial, partial + next_term), max(partial, partial + next_term)

atan5_lo, atan5_hi = arctan_inverse_bounds(5, 60)
atan239_lo, atan239_hi = arctan_inverse_bounds(239, 20)
# Machin: pi = 16 arctan(1/5) - 4 arctan(1/239).
PI_LO = 16*atan5_lo - 4*atan239_hi
PI_HI = 16*atan5_hi - 4*atan239_lo
passed("rational_pi_enclosure", 3 < PI_LO < PI_HI < 4)

upper_bound_rational = T_RATIONAL / (2 * PI_LO)
passed("continuum_bound_below_03089", upper_bound_rational < sp.Rational(3089, 10000))
passed("strict_origin_value", -EPSILON < 0)

print({
    "verified_score_upper": str(sp.N(upper_bound_rational, 18)),
    "claimed_decimal": "0.3089",
    "strict_origin_value": str(-EPSILON),
    "conclusion": "C_SU <= 0.3089",
})


{'verified_score_upper': '0.308887913552750468', 'claimed_decimal': '0.3089', 'strict_origin_value': '-1/1000000', 'conclusion': 'C_SU <= 0.3089'}


**Proof of Theorem 2.1.** Exact recomposition shows that the stored polynomial uses only the self-Fourier modes from Lemma 1.1 and has the stated normalization. The Bernstein calculation proves $P(t)<-10^{-6}$ on $[1.9408,1000]$. Every coefficient of the exact expansion $P(1000+u)$ is negative, so the same inequality continues for $u\ge0$. Lemma 1.2 therefore applies at $T=1.9408$. Finally, the rational Machin-series enclosure proves $1.9408/(2\pi)<0.3089$. $\square$

**Formal proof:** [Lean 4 source](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/sign_uncertainty/formal_proofs/s1).

**Station Reference.** The degree-$226$ witness and whole-ray certificate are from Archive #82 and its addendum, with Research Center Evaluations #1488 and #1513. The notebook strengthens the stored endpoint check by starting at the smaller rational value $1.9408$.

**Related Work.** [Bourgain, Clozel, and Kahane (2010)](https://doi.org/10.5802/aif.2552) introduced the sign-uncertainty problem, and [Gonçalves, Oliveira e Silva, and Steinerberger (2017)](https://doi.org/10.1016/j.jmaa.2017.02.030) developed one-dimensional Hermite-polynomial constructions. [Georgiev et al. (2025)](https://doi.org/10.48550/arXiv.2511.02864) reported the AlphaEvolve value $0.321591$ and summarized an unpublished rigorous bound of $0.3102$. The exact-rational, whole-tail-certified bound $0.3089$ appears to be the strongest publicly documented rigorous upper bound.

<a id="section-3"></a>

## 3. S2. The double-root Laguerre family is exhausted near $0.3153$.

For $4\le k\le20$, let $V_k$ be the span of

$$
L_0^{(-1/2)},L_2^{(-1/2)},\ldots,L_{4k+2}^{(-1/2)}.
$$

A normalized prescribed-double-root polynomial is a $P\in V_k$ with $P(0)=0$, $P'(0)=1$, and prescribed positive numbers $r_1<\cdots<r_k$ satisfying $P(r_i)=P'(r_i)=0$. We call it legal when it has a nonzero odd number of positive sign-changing roots. Its score is $T/(2\pi)$, where $T$ is the final such root; the prescribed double roots are contacts and do not count as sign changes. We write $C_{{\rm DR},20}$ for the infimum of these scores over legal members with $k\le20$.

**Lemma 3.1 (orientation after the last crossing).** If a normalized prescribed-double-root polynomial has an odd number of positive sign changes and final crossing $T$, then $P(t)\le0$ for every $t\ge T$.

**Proof.** Since $P'(0)=1$, the polynomial is positive immediately to the right of zero. Each sign-changing root reverses the sign; the prescribed double roots do not. After an odd number of crossings the sign is therefore nonpositive, with equality allowed only at later even-multiplicity contacts. $\square$

**Theorem 3.2.**

$$
0.315305<C_{{\rm DR},20}<0.3153090099692479.
$$

**Verification.** The next cell reconstructs the explicit $k=20$ polynomial, checks every prescribed double root, counts its positive residual roots exactly, and encloses its score.

In [5]:
def horner(coefficients, point):
    value = sp.Rational(0)
    for coefficient in reversed(coefficients):
        value = value * point + coefficient
    return value

def derivative_coefficients(coefficients):
    return [i * coefficients[i] for i in range(1, len(coefficients))]

def construction_polynomials(record):
    degrees = list(map(int, record["basis_degrees"]))
    roots = [sp.Rational(x) for x in record["root_rationals"]]
    assert degrees == list(range(0, 4 * len(roots) + 3, 2))
    lag = [sp.Rational(x) for x in record["laguerre_coefficients"]]
    power_saved = [sp.Rational(x) for x in record["power_coefficients_ascending"]]
    residual_saved = [sp.Rational(x) for x in record["residual_coefficients_ascending"]]
    power_rebuilt = compose_laguerre(degrees, lag)
    assert power_rebuilt == power_saved
    t = sp.Symbol("t")
    P = sp.Poly.from_list(list(reversed(power_saved)), t, domain=sp.QQ)
    R = sp.Poly.from_list(list(reversed(residual_saved)), t, domain=sp.QQ)
    divisor = sp.Poly(t, t, domain=sp.QQ)
    for root in roots:
        divisor *= sp.Poly((t-root)**2, t, domain=sp.QQ)
    assert P == divisor * R
    return roots, P, R

def pari_integer_polynomial(polynomial):
    coefficients = list(reversed(polynomial.all_coeffs()))
    denominator = sp.ilcm(*[int(value.q) for value in coefficients])
    integers = [int(value * denominator) for value in coefficients]
    common = math.gcd(*integers)
    integers = [value // common for value in integers]
    return PARI.Polrev(integers)

def verify_double_root_record(record, check_root_count=True):
    roots, P, R = construction_polynomials(record)
    P_derivative = P.diff()
    assert P.eval(0) == 0 and P_derivative.eval(0) == 1
    assert all(P.eval(root) == 0 and P_derivative.eval(root) == 0 for root in roots)
    assert all(0 < roots[i] and (i == 0 or roots[i-1] < roots[i]) for i in range(len(roots)))
    pari_residual = pari_integer_polynomial(R)
    if check_root_count:
        assert int(PARI.polsturm(pari_residual, [0, PARI("oo")])) == 1
    assert R.eval(0) > 0 and R.LC() < 0

    recorded_T = sp.Rational(record["station_recorded_score"]["T"])
    if len(roots) == 20:
        # Exact isolating interval persisted with the accepted construction.
        left = sp.Rational(
            "10994725169666489265372696986380479953059/"
            "5549682385733169916106039287298646519682"
        )
        right = sp.Rational(
            "903111898736915434944438961179076978235/"
            "455853522432187348319674192488062242962"
        )
    else:
        radius = sp.Rational(1, 10**11)
        left, right = recorded_T - radius, recorded_T + radius
    assert int(PARI.polsturm(pari_residual, [PARI(str(left)), PARI(str(right))])) == 1
    assert R.eval(left) * R.eval(right) < 0
    score_lower = left / (2 * PI_HI)
    score_upper = right / (2 * PI_LO)
    recorded_score = sp.Rational(record["station_recorded_score"]["upper_bound"])
    if len(roots) == 20:
        assert score_upper < sp.Rational("0.3153090099692479")
    else:
        assert score_lower < recorded_score < score_upper
    return {
        "k": len(roots),
        "degree P": P.degree(),
        "degree residual": R.degree(),
        "positive residual roots": 1,
        "score lower exact": score_lower,
        "score upper exact": score_upper,
        "score lower": float(score_lower),
        "score upper": float(score_upper),
    }

record20 = data["double_root_constructions"]["20"]
upper20 = verify_double_root_record(record20)
passed("double_root_k20_exact_construction")
passed("double_root_k20_upper_bound")
print(upper20)


{'k': 20, 'degree P': 82, 'degree residual': 41, 'positive residual roots': 1, 'score lower': 0.3153090099692479, 'score upper': 0.3153090099692479}


The next cell verifies the exact weighted-sum certificate that excludes every legal construction with $k\le20$ and score at most $0.315305$.

In [6]:
certificate = data["farkas_0315305"]
A0 = sp.Rational(certificate["A_rational"])
nodes = [sp.Rational(x) for x in certificate["nodes"]]
weights = [sp.Rational(x) for x in certificate["weights"]]
mu0, mu1 = [sp.Rational(x) for x in certificate["mu"]]
degrees82 = list(map(int, certificate["basis_degrees"]))
passed("farkas_even_Laguerre_basis", degrees82 == list(range(0, 84, 2)))
basis82 = [laguerre_power_coefficients(degree) for degree in degrees82]

weight_hash = hashlib.sha256("\n".join(map(str, weights)).encode()).hexdigest()
passed("farkas_weight_hash", weight_hash == certificate["source_weight_sha256"])
passed("farkas_41_positive_weights", len(weights) == 41 and all(x > 0 for x in weights))
passed("farkas_derivative_multiplier", mu1 == -1)

# pi < 355/113 is elementary and also follows from our tighter Machin enclosure.
PI_UPPER_SIMPLE = sp.Rational(355, 113)
passed("pi_upper_355_over_113", PI_HI < PI_UPPER_SIMPLE)
threshold_upper = 2 * PI_UPPER_SIMPLE * A0
passed("farkas_nodes_on_tail", all(node > threshold_upper for node in nodes))

identity_residuals = []
for basis_polynomial in basis82:
    node_sum = sum(
        weight * horner(basis_polynomial, node)
        for weight, node in zip(weights, nodes)
    )
    value_at_zero = basis_polynomial[0]
    derivative_at_zero = basis_polynomial[1] if len(basis_polynomial) > 1 else 0
    identity_residuals.append(node_sum + mu0*value_at_zero + mu1*derivative_at_zero)
passed("farkas_exact_identity", all(value == 0 for value in identity_residuals))

print({
    "basis": "$L_0,L_2,\\ldots,L_{82}$",
    "dimension": len(degrees82),
    "tail nodes": len(nodes),
    "all weights": "strictly positive",
    "smallest node margin": str(min(nodes)-threshold_upper),
    "identity residual": "exactly zero",
    "weight SHA-256": weight_hash[:16] + "…",
})


{'basis': '$L_0,L_2,\\ldots,L_{82}$', 'dimension': 42, 'tail nodes': 41, 'all weights': 'strictly positive', 'smallest node margin': '1/11300000', 'identity residual': 'exactly zero', 'weight SHA-256': 'a59cf51fe04f8e28…'}


**Proof of Theorem 3.2.** The first verification reconstructs an explicit $k=20$ polynomial exactly in the even Laguerre basis. It verifies the normalization and all twenty double-root conditions, and exact real-root counting shows that the degree-$41$ residual has exactly one positive root. Rational root and $\pi$ enclosures give the stated upper bound.

For the lower bound, let $s_1,\ldots,s_{41}$ and $\lambda_1,\ldots,\lambda_{41}$ be the exact nodes and positive weights checked in the second verification. The verified basis identities imply, for every $P\in\operatorname{span}\{L_0,L_2,\ldots,L_{82}\}$ with $P(0)=0$ and $P'(0)=1$,

$$
\sum_{i=1}^{41}\lambda_iP(s_i)=1.
$$

Every node satisfies $s_i>2\pi(0.315305)$. If a legal construction had score at most $0.315305$, Lemma 3.1 would give $P(s_i)\le0$ for every $i$, contradicting the identity. Finally, each space with $k<20$ embeds in the degree-$82$ space by setting its higher Laguerre coefficients to zero. $\square$

**Formal proof:** [Lean 4 source](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/sign_uncertainty/formal_proofs/s2).

**Station Reference.** The accepted $k=20$ construction is from Evaluation #1711. The exact weighted-sum data are from Evaluation #832 and are developed further in Archives #18 and #42; the bridge to every legal $k\le20$ construction is audited in Archive #105 and Evaluations #1821, #1824, #1826, #1829, #1832, and #1835.

**Related Work.** [Cohn and Gonçalves (2019), Proposition 5.1 and Corollary 5.2](https://doi.org/10.1007/s00222-019-00875-4) used a summation formula at the final crossing and prescribed double roots to certify optimality in a fixed Laguerre span; their dimension-one computations reported $k=1,2,5$ and used floating-point arithmetic. The certificate above gives an exact-rational dimension-one result at $k=20$, and one degree-$82$ identity simultaneously covers every $k\le20$. [Georgiev et al. (2025)](https://doi.org/10.48550/arXiv.2511.02864) later used the prescribed-double-root Laguerre family and reported $0.321591$.

<a id="section-4"></a>

## 4. Additional findings

These are additional results that may be interesting but are not included in the spotlight.

<a id="section-4-1"></a>

### 4.1 Exact constructions beyond the certified $k\le20$ range

To determine whether the $k\le20$ barrier persists at higher dimension, we construct exact examples with exactly one positive residual root for $k=21,22,23,24$, all with scores below $0.315305$. Their scores decrease to $0.3142477184$, proving that the capped lower barrier disappears as soon as the representation is enlarged. The four constructions show that enlarging the prescribed-root family opens feasible sign topologies below the $k\le20$ barrier.

**Theorem 4.1.** The prescribed-double-root family admits explicit examples with exactly one positive residual root at $k=21,22,23,24$. Their scores are:

$$
\begin{array}{c|cccc}
k&21&22&23&24\\ \hline
T/(2\pi)&0.3147530191&0.3146045685&0.3143408810&0.3142477184.
\end{array}
$$

Thus the $0.315305$ barrier for $k\le20$ disappears when the representation dimension increases.

**Verification.** The next cell reconstructs and verifies all four exact double-root constructions and checks that their score intervals decrease and lie below $0.315305$.

In [7]:
ladder_rows = []
for k in range(21, 25):
    row = verify_double_root_record(data["double_root_constructions"][str(k)])
    ladder_rows.append(row)
    passed(f"double_root_k{k}_exact_construction")

for earlier, later in zip(ladder_rows, ladder_rows[1:]):
    assert later["score upper exact"] < earlier["score lower exact"]
assert all(row["score upper exact"] < sp.Rational(315305, 10**6) for row in ladder_rows)
passed("post_cap_ladder_monotone_and_below_floor")
print(f"{'k':>3} {'degree P':>9} {'degree residual':>16} {'score upper':>18}")
for row in ladder_rows:
    print(f"{row['k']:3d} {row['degree P']:9d} {row['degree residual']:16d} {row['score upper']:18.12f}")


  k  degree P  degree residual        score upper
 21        86               43     0.314753019139
 22        90               45     0.314604568453
 23        94               47     0.314340881029
 24        98               49     0.314247718425


**Proof of Theorem 4.1.** For each row, the code independently recomposes the polynomial from exact Laguerre coefficients, checks all normalization and double-root identities, divides the prescribed factors exactly, and applies exact real-root counting to the residual. Each residual has exactly one positive root, positive sign at the origin, and negative leading coefficient. The rational enclosures printed above therefore enclose the final sign-change scores. $\square$

**Station Reference.** The beyond-cap ladder is from Archive #62 and Research Center Evaluations #1126, #1141, and #1142.

**Related Work.** The prescribed-double-root Laguerre construction was introduced by [Cohn and Gonçalves (2019, §4)](https://doi.org/10.1007/s00222-019-00875-4), whose published dimension-one computations report $k=1,2,5$ and describe numerical instability in low dimensions. [Georgiev et al. (2025)](https://doi.org/10.48550/arXiv.2511.02864) report searches for $k=6,\ldots,12$, reaching $C_{\rm SU}\le0.321591$ at $k=12$. Section 3 certifies the full family through $k=20$, while the exact constructions above extend the verified dimension-one range to $k=21,22,23,24$.

<a id="section-4-2"></a>

### 4.2 Analytic chambers of prescribed double roots

We describe the geometry of the prescribed-double-root parameter space by proving that it decomposes into analytic chambers with constant residual-root order and sign topology. Within each chamber, every simple residual root varies analytically with the prescribed roots, and the final crossing satisfies an explicit derivative formula. A feasible topology can therefore change only at one of four algebraic walls: a residual-root collision, a degree drop, a singular reconstruction matrix, or a collision of prescribed roots with one another or the origin. This converts the observed changes in sign pattern into a precise wall-crossing theory and supplies a rigorous foundation for continuation methods in the double-root model.

Put $\ell_j(t)=L_{2j}^{(-1/2)}(t)$ for $0\le j\le2k+1$. For $r=(r_1,\ldots,r_k)$, let $M(r)$ be the $(2k+2)\times(2k+2)$ matrix whose rows are

$$
(\ell_j(0))_j,\quad(\ell_j'(0))_j,\quad(\ell_j(r_i))_j,\quad(\ell_j'(r_i))_j\qquad(1\le i\le k).
$$

The coefficient vector of $P(t;r)=\sum_{j=0}^{2k+1}c_j\ell_j(t)$ solves $M(r)c=(0,1,0,\ldots,0)^{\mathsf T}$. Define

$$
U_k=\{0<r_1<\cdots<r_k:\det M(r)\ne0\}.
$$

For $r\in U_k$, let

$$
N(t;r)=\frac{P(t;r)}{t\prod_{i=1}^k(t-r_i)^2},
$$

which has degree at most $2k+1$.

**Theorem 4.2 (analytic chamber theorem).** Remove from $U_k$ the points where the discriminant of $N$ vanishes or its leading coefficient vanishes. On each connected component of the remaining set:

1. the number and order of real residual roots are constant;
2. every real residual root varies real analytically with $r$;
3. the positive sign-change count is constant; and
4. if $T(r)$ is the final crossing, then

   $$
   \frac{\partial T}{\partial r_i}
   =-\frac{\partial_{r_i}N(T;r)}{\partial_tN(T;r)}.
   $$

**Proof.** The entries of $M(r)$ are polynomial functions of $r$, so Cramer's rule makes the coefficients of $P$ and $N$ rational, hence analytic, on $U_k$. Off the discriminant, all roots are simple. The implicit-function theorem then continues each real root analytically and gives the derivative formula. A simple real root can leave the real axis only by colliding with another root, which would make the discriminant vanish. It cannot cross the origin because the normalization gives

$$
N(0;r)=\frac{1}{\prod_i r_i^2}>0.
$$

The leading-coefficient exclusion prevents a root from escaping through infinity. Thus the order and positive sign-change count remain constant throughout each component. $\square$

The theorem identifies the generic routes between feasible topologies: a residual-root collision, a degree drop, a singular reconstruction, or a collision of prescribed-root coordinates with one another or the origin at the boundary of $U_k$.

**Station Reference.** The chamber formulation is from Archive #49 and Research Center Evaluations #949 and #963.

**Related Work.** Discriminant decompositions and semialgebraic chambers are classical; see [Basu, Pollack, and Roy (2006)](https://doi.org/10.1007/3-540-33099-2). [Cohn and Gonçalves (2019, §4)](https://doi.org/10.1007/s00222-019-00875-4) identify double-root splitting, leading-coefficient vanishing, and extra double roots as sources of discontinuity in the prescribed-double-root Laguerre model. Theorem 4.2 organizes these mechanisms and the remaining boundary phenomena into analytic chambers with constant sign topology and an explicit continuation derivative.